# EPIC Clarity Provider Hydration

This notebook hydrates the OMOP PROVIDER table from EPIC Clarity source data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.CLARITY_SER` - Provider/clinician master

## OMOP Fields Populated
- provider_source_value
- provider_name
- provider_id (surrogate key from mapping table)

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_provider_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_SER', 'PROV_ID', s.PROV_ID) AS provider_source_value,
    s.PROV_NAME AS provider_name,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.clarity_ser s
WHERE s.PROV_ID IS NOT NULL
''')

display(silver_provider_df)
silver_provider_df.createOrReplaceTempView("silver_provider")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.provider AS target
USING silver_provider AS source
ON target.provider_source_value = source.provider_source_value

WHEN MATCHED AND NOT (
    target.provider_name <=> source.provider_name
)
THEN UPDATE SET
    target.provider_name = source.provider_name,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    provider_source_value,
    provider_name,
    updated_tsp
)
VALUES (
    source.provider_source_value,
    source.provider_name,
    source.updated_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_provider (
    source_system,
    provider_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.provider_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.updated_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, provider_source_value, updated_tsp
    FROM _exponent.omop_silver.provider
    WHERE provider_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_provider x
    ON s.provider_source_value = x.provider_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
gold_provider_df = spark.sql("""
SELECT
    m.provider_id,
    s.provider_name,
    s.updated_tsp
FROM _exponent.omop_silver.provider s
INNER JOIN _exponent.omop_mapping.source_to_provider m
    ON s.provider_source_value = m.provider_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
""")

display(gold_provider_df)
gold_provider_df.createOrReplaceTempView("gold_provider")

In [ ]:
%sql
MERGE INTO _exponent.omop.provider AS target
USING gold_provider AS source
ON target.provider_id = source.provider_id

WHEN MATCHED AND NOT (
    target.provider_name <=> source.provider_name
)
THEN UPDATE SET
    target.provider_name = source.provider_name

WHEN NOT MATCHED THEN INSERT (
    provider_id,
    provider_name
)
VALUES (
    source.provider_id,
    source.provider_name
)